In [ ]:
import subprocess, sys

def _pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])

_pip("numpy==1.26.4")                   # pin first — prevents binary mismatch with pandas

_pip(
    "transformers==4.44.0",
    "datasets==2.20.0",
    "peft==0.12.0",
    "trl==0.9.6",
    "bitsandbytes>=0.44.0",             # ← was 0.43.3; fixes triton.ops crash
    "accelerate==0.33.0",
    "torch",
    "huggingface_hub",
    "pandas==2.2.2",
    "matplotlib",
)

✅ Install done — RESTART the runtime, then run from Cell 2 onwards.


In [ ]:
# ── CELL 2 · Imports & sanity checks ─────────────────────────────────────────

import numpy as np
import pandas as pd
import torch
import time
import re
import json
from pathlib import Path
from datasets import Dataset

import matplotlib
matplotlib.use("Agg")          # non-interactive backend (works in Colab too)
import matplotlib.pyplot as plt

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

In [ ]:
from huggingface_hub import login
login()          # prompts for token interactively; or pass token= kwarg

In [ ]:
BASE_MODEL   = "inceptionai/jais-family-6p7b-chat"
DATASET_FILE = "ibn_battuta_arabic_dataset.jsonl"   # path to the dataset
RESULTS_DIR  = Path("results"); RESULTS_DIR.mkdir(exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU   : {torch.cuda.get_device_name(0)}")
    print(f"VRAM  : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Device: cuda
GPU   : Tesla T4
VRAM  : 15.6 GB


In [ ]:
ALL_MODULES  = "auto"
ATTN_ONLY    = "attn"

# Shared SFT training config
SHARED_TRAIN_CONFIG = dict(
    num_train_epochs      = 3,
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 4,
    learning_rate         = 2e-4,
    fp16                  = (DEVICE == "cuda"),
    logging_steps         = 5,
    max_seq_length        = 512,
    warmup_ratio          = 0.05,
    lr_scheduler_type     = "cosine",
    save_strategy         = "no",
    report_to             = "none",
)

In [ ]:
def load_jsonl(path: str) -> Dataset:
    rows = [json.loads(l) for l in open(path, encoding="utf-8")]
    return Dataset.from_list(rows)

dataset = load_jsonl(DATASET_FILE)
print(f"Dataset loaded: {len(dataset)} examples")

# Few-shot subsets for Experiment E
few_shot_datasets = {
    "k=3":  dataset.select(range(3)),
    "k=5":  dataset.select(range(5)),
    "k=10": dataset.select(range(10)),
}
train_full = dataset      # all examples for main experiments

Dataset loaded: 29 examples


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token    = tokenizer.eos_token
tokenizer.padding_side = "right"

sample_ar = "رحلت إلى مملكة مالي عابراً الصحراء الكبرى من المغرب"
sample_en = "I traveled to the kingdom of Mali crossing the Sahara from Morocco"
tok_ar = tokenizer(sample_ar)["input_ids"]
tok_en = tokenizer(sample_en)["input_ids"]
print(f"Arabic  ({len(sample_ar.split())} words) → {len(tok_ar)} tokens  "
      f"({len(tok_ar)/len(sample_ar.split()):.2f} tok/word)")
print(f"English ({len(sample_en.split())} words) → {len(tok_en)} tokens  "
      f"({len(tok_en)/len(sample_en.split()):.2f} tok/word)")
print("Jais tokenises Arabic ~2× more efficiently than Llama-3 would.")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Arabic  (9 words) → 12 tokens  (1.33 tok/word)
English (12 words) → 14 tokens  (1.17 tok/word)
Jais tokenises Arabic ~2× more efficiently than Llama-3 would.


In [ ]:
ATTN_ONLY   = "attn"

In [ ]:
def load_base_model(for_inference: bool = False):
    """Load Jais-6.7b with 4-bit QLoRA quantisation."""
    bnb = BitsAndBytesConfig(
        load_in_4bit              = True,
        bnb_4bit_quant_type       = "nf4",
        bnb_4bit_compute_dtype    = torch.bfloat16,
        bnb_4bit_use_double_quant = True,
    )
    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        quantization_config = bnb,
        device_map          = "auto",
        trust_remote_code   = True,
    )
    model.config.use_cache = for_inference
    if not for_inference:
        # use_gradient_checkpointing=False avoids the in-place *= on Jais
        # embeddings_scale that triggers "leaf Variable used in in-place op".
        model = prepare_model_for_kbit_training(
            model, use_gradient_checkpointing=False
        )
    return model


In [ ]:
def _resolve_target_modules(model, hint: str) -> list:
    """
    Detect actual nn.Linear leaf names from the loaded model.
    hint='auto'  → all linear leaves (excluding lm_head)
    hint='attn'  → only leaves whose name contains 'attn'
    hint=list    → passed through unchanged (explicit override)
    """
    if isinstance(hint, list):
        return hint

    leaf_names = set()
    for name, module in model.named_modules():
        if isinstance(module, torch.nn.Linear):
            leaf = name.split(".")[-1]
            if leaf != "lm_head":
                leaf_names.add(leaf)

    # lm_head operates on full-precision logits — exclude from LoRA.
    leaf_names.discard("lm_head")
    print(f"  All linear leaves found : {sorted(leaf_names)}")

    if hint == "attn":
        attn_kw = ("attn", "qkv", "query", "key", "value")
        selected = [n for n in leaf_names if any(k in n.lower() for k in attn_kw)]
        selected = selected or list(leaf_names)   # fallback: use all if none match
    else:  # "auto"
        selected = list(leaf_names)

    print(f"  LoRA target modules     : {sorted(selected)}")
    return selected


def apply_lora(model, target_modules_hint, frozen_layers=None):
    """
    Wrap model with LoRA adapters.
    target_modules_hint: 'auto', 'attn', or an explicit list of module names.
    Optionally freezes LoRA params in the specified transformer block indices.
    """
    resolved = _resolve_target_modules(model, target_modules_hint)

    lora_cfg = LoraConfig(
        r              = 16,
        lora_alpha     = 32,
        target_modules = resolved,
        lora_dropout   = 0.05,
        bias           = "none",
        task_type      = "CAUSAL_LM",
    )
    model = get_peft_model(model, lora_cfg)

    if frozen_layers:
        for name, param in model.named_parameters():
            if "lora_" in name:
                for idx in frozen_layers:
                    if f".{idx}." in name:
                        param.requires_grad = False

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in model.parameters())
    print(f"  Trainable params: {trainable:,}  ({100*trainable/total:.2f}%)")
    return model

In [ ]:
ARABIC_STYLE_MARKERS = [
    "دخلت", "رحلت", "وصلت", "غادرت", "شاهدت", "لقيت",
    "السلطان", "الرحلة", "الحج", "بلاد", "أهل",
    "أعترف", "في زماني", "رأيت", "حُدِّثت",
    "والله", "سبحانه", "حمداً لله", "بفضل الله",
    "عجيب", "غريب", "مدهش", "بهر",
]

ARABIC_EVAL_QUESTIONS = [
    "صف لي أسواق القاهرة في عصرك.",
    "كيف كان بلاط السلطان في دلهي؟",
    "صف الرحلة البحرية من الهند إلى الصين.",
    "ما نصيحتك لمسافر شاب؟",
    "قارن بين كرم الهند وكرم أهل بلاد القرم.",
]

In [ ]:
def compute_arabic_style_score(text: str) -> float:
    """Fraction of style markers present in the response (0–1)."""
    hits = sum(1 for m in ARABIC_STYLE_MARKERS if m in text)
    return hits / len(ARABIC_STYLE_MARKERS)


def generate_response(model, question: str, max_new_tokens: int = 256) -> str:
    """Generate an Ibn Battuta-style Arabic response."""
    prompt = (
        "أنت ابن بطوطة الرحّالة المغربي.\n"
        f"سؤال: {question}\nجواب:"
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens = max_new_tokens,
            do_sample      = True,
            temperature    = 0.7,
            top_p          = 0.9,
            pad_token_id   = tokenizer.eos_token_id,
        )
    return tokenizer.decode(
        out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
    ).strip()


def evaluate_model(model, label: str) -> dict:
    """Compute perplexity + style score on ARABIC_EVAL_QUESTIONS."""
    model.eval()
    responses  = [generate_response(model, q) for q in ARABIC_EVAL_QUESTIONS]
    style_avg  = sum(compute_arabic_style_score(r) for r in responses) / len(responses)

    # Perplexity on a fixed Arabic passage
    ref = "رحلت إلى مملكة مالي سنة اثنتين وخمسين وسبعمائة للهجرة عابراً الصحراء الكبرى"
    enc = tokenizer(ref, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        loss = model(**enc, labels=enc["input_ids"]).loss
    ppl = torch.exp(loss).item()

    print(f"  [{label}]  PPL={ppl:.2f}  style={style_avg:.3f}")
    return {"label": label, "responses": responses,
            "perplexity": round(ppl, 2), "style_score": round(style_avg, 3)}

In [ ]:
def run_experiment(name, train_dataset,
                   target_modules="auto",
                   frozen_layers=None,
                   extra_config=None):
    """Load model, apply LoRA, train, evaluate, return result dict."""
    print(f"\n{'='*60}\n🔬 {name}\n{'='*60}")
    print(f"  Training examples : {len(train_dataset)}")
    print(f"  Frozen layers     : {frozen_layers}")

    output_dir = str(RESULTS_DIR / name.lower().replace(" ", "_").replace(":", ""))
    cfg = {**SHARED_TRAIN_CONFIG, **(extra_config or {})}

    torch.cuda.reset_peak_memory_stats()
    t0 = time.time()

    model = load_base_model()
    model = apply_lora(model, target_modules, frozen_layers)

    sft_cfg = SFTConfig(output_dir=output_dir, **cfg)
    trainer = SFTTrainer(
        model           = model,
        train_dataset   = train_dataset,
        tokenizer       = tokenizer,
        args            = sft_cfg,
    )
    trainer.train()

    elapsed_min  = (time.time() - t0) / 60
    peak_vram_gb = (torch.cuda.max_memory_allocated() / 1e9
                    if DEVICE == "cuda" else 0.0)

    metrics = evaluate_model(model, name)
    del model; torch.cuda.empty_cache()

    return {
        "experiment"    : name,
        "n_train"       : len(train_dataset),
        "train_time_min": round(elapsed_min, 1),
        "peak_vram_gb"  : round(peak_vram_gb, 2),
        **metrics,
    }

In [ ]:
bnb_inf = BitsAndBytesConfig(load_in_4bit=True,
                              bnb_4bit_compute_dtype=torch.bfloat16)
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, quantization_config=bnb_inf,
    device_map="auto", trust_remote_code=True
)
base_model.config.use_cache = True

FEW_SHOT_AR = [
    {"q": "من أنت؟",
     "a": "أنا أبو عبد الله محمد بن بطوطة الطنجي. طفت ما يزيد على خمسة وسبعين ألف ميل وجمعت ما رأيت في كتابي تحفة النظار."},
    {"q": "صف القسطنطينية.",
     "a": "القسطنطينية مدينة عظيمة يجتمع فيها بحران وتسمو فيها كنيسة آيا صوفيا بقبة تُذهل العقول."},
]

def few_shot_prompt(question: str) -> str:
    prefix = "".join(
        f"سؤال: {ex['q']}\nجواب: {ex['a']}\n\n" for ex in FEW_SHOT_AR
    )
    return prefix + f"سؤال: {question}\nجواب:"

baseline_responses = [
    generate_response(base_model, q) for q in ARABIC_EVAL_QUESTIONS
]
baseline_style = sum(
    compute_arabic_style_score(r) for r in baseline_responses
) / len(baseline_responses)
print(f"Few-shot baseline style score: {baseline_style:.3f}")

results = []
results.append({
    "experiment"    : "A: Few-shot baseline",
    "n_train"       : 0,
    "train_time_min": 0,
    "peak_vram_gb"  : 0,
    "responses"     : baseline_responses,
    "perplexity"    : None,
    "style_score"   : round(baseline_style, 3),
})

del base_model; torch.cuda.empty_cache()

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:391: UserWarning: Some matrices hidden dimension is not a multiple of 64 and efficient inference kernels are not supported for these (slow). Matrix input size found: torch.Size([1, 1, 10928])
  warn(


Few-shot baseline style score: 0.052


In [ ]:
results.append(run_experiment(
    "B: Full LoRA",
    train_full, ALL_MODULES, frozen_layers=None
))



🔬 B: Full LoRA
  Training examples : 29
  Frozen layers     : None


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

  All linear leaves found : ['c_attn', 'c_fc', 'c_fc2', 'c_proj']
  LoRA target modules     : ['c_attn', 'c_fc', 'c_fc2', 'c_proj']
  Trainable params: 35,659,776  (0.99%)


/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:366: UserWarning: You passed a `dataset_kwargs` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(


Map:   0%|          | 0/29 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:488: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


Step,Training Loss
5,3.397600
10,1.895500
15,0.984100
20,0.744500


/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:391: UserWarning: Some matrices hidden dimension is not a multiple of 64 and efficient inference kernels are not supported for these (slow). Matrix input size found: torch.Size([1, 1, 10928])
  warn(


  [B: Full LoRA]  PPL=518.76  style=0.043


In [ ]:
results.append(run_experiment(
    "C1: Top-half LoRA (freeze 0-15)",
    train_full, ALL_MODULES, frozen_layers=list(range(16))
))

results.append(run_experiment(
    "C2: Top-quarter LoRA (freeze 0-23)",
    train_full, ALL_MODULES, frozen_layers=list(range(24))
))


🔬 C1: Top-half LoRA (freeze 0-15)
  Training examples : 29
  Frozen layers     : [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

  All linear leaves found : ['c_attn', 'c_fc', 'c_fc2', 'c_proj']
  LoRA target modules     : ['c_attn', 'c_fc', 'c_fc2', 'c_proj']
  Trainable params: 17,829,888  (0.49%)


/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:366: UserWarning: You passed a `dataset_kwargs` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(


Map:   0%|          | 0/29 [00:00<?, ? examples/s]

...

Map:   0%|          | 0/29 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:488: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


Step,Training Loss
5,3.632200
10,2.837500
15,2.038400
20,1.709800


/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:391: UserWarning: Some matrices hidden dimension is not a multiple of 64 and efficient inference kernels are not supported for these (slow). Matrix input size found: torch.Size([1, 1, 10928])
  warn(


  [C2: Top-quarter LoRA (freeze 0-23)]  PPL=629.74  style=0.035


In [ ]:
results.append(run_experiment(
    "D: Head-only (last 4 blocks, attn)",
    train_full, ATTN_ONLY, frozen_layers=list(range(28))
))


🔬 D: Head-only (last 4 blocks, attn)
  Training examples : 29
  Frozen layers     : [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27]


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

  All linear leaves found : ['c_attn', 'c_fc', 'c_fc2', 'c_proj']
  LoRA target modules     : ['c_attn']
  Trainable params: 1,048,576  (0.03%)


/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:366: UserWarning: You passed a `dataset_kwargs` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(


Map:   0%|          | 0/29 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:488: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


Step,Training Loss
5,3.744900
10,3.643600
15,3.454600
20,3.381600


/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:391: UserWarning: Some matrices hidden dimension is not a multiple of 64 and efficient inference kernels are not supported for these (slow). Matrix input size found: torch.Size([1, 1, 10928])
  warn(


  [D: Head-only (last 4 blocks, attn)]  PPL=833.05  style=0.043


In [ ]:
for name, ds in few_shot_datasets.items():
    k = len(ds)
    results.append(run_experiment(
        f"E: Few-shot {name} LoRA",
        ds, ALL_MODULES, frozen_layers=None,
        extra_config={"num_train_epochs": max(10, 60 // k)},
    ))


🔬 E: Few-shot k=3 LoRA
  Training examples : 3
  Frozen layers     : None


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

  All linear leaves found : ['c_attn', 'c_fc', 'c_fc2', 'c_proj']
  LoRA target modules     : ['c_attn', 'c_fc', 'c_fc2', 'c_proj']
  Trainable params: 35,659,776  (0.99%)


/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:366: UserWarning: You passed a `dataset_kwargs` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(


Map:   0%|          | 0/3 [00:00<?, ? examples/s]

...

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:488: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


Step,Training Loss
5,3.318700
10,1.793200
15,0.966800
20,0.730600


/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:391: UserWarning: Some matrices hidden dimension is not a multiple of 64 and efficient inference kernels are not supported for these (slow). Matrix input size found: torch.Size([1, 1, 10928])
  warn(


  [E: Few-shot k=10 LoRA]  PPL=561.90  style=0.035


In [ ]:
COLS = ["experiment", "n_train", "perplexity", "style_score",
        "train_time_min", "peak_vram_gb"]
df = pd.DataFrame([{c: r[c] for c in COLS} for r in results])
df_sorted = df.sort_values("style_score", ascending=False).reset_index(drop=True)
print("\n" + df_sorted.to_string(index=False))
df_sorted.to_csv(RESULTS_DIR / "comparison_arabic.csv", index=False)


                        experiment  n_train  perplexity  style_score  train_time_min  peak_vram_gb
              A: Few-shot baseline        0         NaN        0.052             0.0          0.00
                      B: Full LoRA       29      518.76        0.043             4.3         14.04
D: Head-only (last 4 blocks, attn)       29      833.05        0.043             3.5          8.13
              E: Few-shot k=3 LoRA        3      594.09        0.043             3.3         14.04
              E: Few-shot k=5 LoRA        5      656.20        0.043             3.5         14.06
   C1: Top-half LoRA (freeze 0-15)       29      610.60        0.035             4.0         10.76
C2: Top-quarter LoRA (freeze 0-23)       29      629.74        0.035             3.8          9.12
             E: Few-shot k=10 LoRA       10      561.90        0.035             4.4         14.06


In [ ]:
colors = plt.cm.tab10(range(len(df_sorted)))

# Chart 1: PPL vs Style score
df_plot = df_sorted.dropna(subset=["perplexity"])
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("Ibn Battuta (Arabic) — Fine-Tuning Comparison · Jais-6.7b",
             fontsize=13, fontweight="bold")

ax = axes[0]
for i, row in df_plot.iterrows():
    ax.scatter(row["perplexity"], row["style_score"], color=colors[i], s=130, zorder=3)
    ax.annotate(row["experiment"].split(":")[0],
                (row["perplexity"], row["style_score"]),
                textcoords="offset points", xytext=(5, 5), fontsize=8)
ax.set_xlabel("Perplexity ↓"); ax.set_ylabel("Style Score ↑")
ax.set_title("PPL vs Style Score"); ax.grid(alpha=0.3)

# Chart 2: Style score bar
ax = axes[1]
ax.barh(df_sorted["experiment"], df_sorted["style_score"], color=colors)
ax.set_xlabel("Style Score ↑"); ax.set_title("Style Score by Experiment")
ax.grid(axis="x", alpha=0.3)


# Chart 3: Few-shot learning curve
fs = sorted(
    [r for r in results if "Few-shot E" in r["experiment"]],
    key=lambda r: r["n_train"]
)
if len(fs) >= 2:
    fig2, ax2 = plt.subplots(figsize=(8, 4))
    ax2.plot([r["n_train"] for r in fs], [r["style_score"] for r in fs],
             "o-", linewidth=2.5, markersize=9, color="darkorchid")
    for r in fs:
        ax2.annotate(r["experiment"], (r["n_train"], r["style_score"]),
                     textcoords="offset points", xytext=(4, 4), fontsize=8)
    ax2.set_xlabel("Training examples (k)"); ax2.set_ylabel("Style Score ↑")
    ax2.set_title("Few-Shot Learning Curve"); ax2.grid(alpha=0.3)
    plt.tight_layout()
    fig2.savefig(RESULTS_DIR / "few_shot_curve.png", dpi=150, bbox_inches="tight")
    print("Saved: few_shot_curve.png")


In [ ]:
Q_IDX = 0   # change 0-4 to inspect different evaluation questions
print(f"\nالسؤال: {ARABIC_EVAL_QUESTIONS[Q_IDX]}\n" + "="*65)
for r in results:
    resp = r["responses"][Q_IDX]
    print(f"\n [{r['experiment']}]")
    print(f"   الجواب: {resp[:400]}...")
    print("-"*65)


السؤال: صف لي أسواق القاهرة في عصرك.

 [A: Few-shot baseline]
   الجواب: كانت القاهرة مدينة مزدحمة في عصر ابن بطوطة، وكانت أسواقها تعج بالباعة والمشترين. وكانت تبيع مختلف السلع، من الأقمشة والأقمشة الحريرية إلى التوابل والحبوب، وحتى الذهب والفضة. وكان هناك أيضا سوق خاص للخياطين، حيث كان الخياطون يبيعون ملابسهم المصنوعة يدويا. وكانت هناك أيضا أماكن تبيع الكتب والمستندات، مثل الكتّاب، حيث كان الناس يشترون الكتب والمستندات الورقية. وكانت بعض الأسواق تبيع الحيوانات الحية، ...
-----------------------------------------------------------------

 [B: Full LoRA]
   الجواب: كانت القاهرة مدينة مزدحمة جداً في عصر الرحّالة المغربي ابن بطوطة. كانت الأسواق تعج بالباعة والمشترين. كانت هناك أسواق متخصصة لكل شيء من الأقمشة والخزف إلى التوابل والعطارة. وكانت هناك أيضاً أسواق مفتوحة حيث يبيع التجار بضائعهم على الطاولات. وكانت الشوارع مزدحمة بالمارة والعربات التي تجرها الخيول والجمال. وكانت هناك أيضاً أسواق مؤقتة أقامها التجار الذين كانوا يسافرون من أماكن بعيدة لبيع بضائعهم....
---------------------------